In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/Studia-II/Uczenie-Maszynowe/Lista5/Zad1

SKOPIOWANE Z ZAD5 LISTA 4, niżesz zaczyna się zad1

In [3]:
df = pd.read_csv('../data/Titanic Dataset.csv')

In [4]:
# wybieramy tylko sensowne zmienne kategoryczne
cat_cols = ['sex', 'pclass', 'embarked']

num_cols = df.select_dtypes(include=['number']).columns

print("PROCENTY BRAKÓW")

for num_col in num_cols:
    if df[num_col].isnull().sum() > 0:
        print(f"\n=== Analiza braków w kolumnie '{num_col}' ===")
        for cat_col in cat_cols:
            grouped = df.groupby(cat_col)[num_col].apply(lambda x: x.isnull().mean() * 100)
            print(grouped)


PROCENTY BRAKÓW

=== Analiza braków w kolumnie 'age' ===
sex
female    16.738197
male      21.945433
Name: age, dtype: float64
pclass
1    12.074303
2     5.776173
3    29.337094
Name: age, dtype: float64
embarked
C    21.481481
Q    59.349593
S    14.442013
Name: age, dtype: float64

=== Analiza braków w kolumnie 'fare' ===
sex
female    0.000000
male      0.118624
Name: fare, dtype: float64
pclass
1    0.000000
2    0.000000
3    0.141044
Name: fare, dtype: float64
embarked
C    0.000000
Q    0.000000
S    0.109409
Name: fare, dtype: float64

=== Analiza braków w kolumnie 'body' ===
sex
female    98.283262
male      86.595492
Name: body, dtype: float64
pclass
1    89.164087
2    88.808664
3    92.242595
Name: body, dtype: float64
embarked
C    90.740741
Q    94.308943
S    90.262582
Name: body, dtype: float64


In [5]:
df['sex'] = df['sex'].map({'male': 1, 'female': 0}) # od razu

In [6]:
missing_percentage = df.isnull().mean() * 100
print(missing_percentage.sort_values(ascending=False))

body         90.756303
cabin        77.463713
boat         62.872422
home.dest    43.086325
age          20.091673
embarked      0.152788
fare          0.076394
sibsp         0.000000
name          0.000000
survived      0.000000
pclass        0.000000
sex           0.000000
parch         0.000000
ticket        0.000000
dtype: float64


In [7]:
print(df['boat'].value_counts()) # nie ma sensu tutaj zamieniać braków modą

boat
13         39
C          38
15         37
14         33
4          31
10         29
5          27
3          26
9          25
11         25
8          23
16         23
7          23
D          20
6          20
12         19
2          13
A          11
B           9
1           5
13 15       2
5 7         2
C D         2
5 9         1
8 10        1
13 15 B     1
15 16       1
Name: count, dtype: int64


In [8]:
df_base = df.copy()
cols_to_drop_1 = ['name', 'body', 'cabin', 'boat', 'home.dest', 'ticket']

df_base['age'] = df_base['age'].fillna(df_base['age'].mean())
df_base['fare'] = df_base['fare'].fillna(df_base['fare'].mean())
df_base['embarked'] = df_base['embarked'].fillna(df_base['embarked'].mode()[0])

df_base.drop(columns=cols_to_drop_1, inplace=True)

# One-hot encoding
cat_features = df_base.select_dtypes(exclude=['number']).columns
df_base = pd.get_dummies(df_base, columns=cat_features, drop_first=True, dtype=int)

print(f"Pozostałe braki w df_base: {df_base.isnull().sum().sum()}")

Pozostałe braki w df_base: 0


In [9]:
# display(df_base)

In [10]:
df_adv = df.copy()

df_adv['has_cabin'] = df_adv['cabin'].notnull().astype(int)

# Wyodrębnienie tytułu z imienia
df_adv['title'] = df_adv['name'].str.extract(r' ([A-Za-z]+)\.', expand=False)

rare_titles = df_adv['title'].value_counts()[df_adv['title'].value_counts() < 10].index
df_adv['title'] = df_adv['title'].replace(list(rare_titles), 'Rare')

# Uzupełnianie wieku medianą w zależności od tytułu
median_by_title = df_adv.groupby('title')['age'].median()
df_adv['age'] = df_adv.apply(
    lambda row: median_by_title[row['title']] if pd.isnull(row['age']) else row['age'],
    axis=1
)
# Jeśli coś jeszcze zostało puste, uzupełnij medianą ogólną
df_adv['age'] = df_adv['age'].fillna(df_adv['age'].median())

# Pozostałe braki
df_adv['fare'] = df_adv['fare'].fillna(df_adv['fare'].mean())
df_adv['embarked'] = df_adv['embarked'].fillna(df_adv['embarked'].mode()[0])

# Usuwanie zbędnych kolumn
cols_to_drop_2 = ['name', 'body', 'boat', 'home.dest', 'ticket', 'cabin']
df_adv.drop(columns=cols_to_drop_2, inplace=True)

# One-hot encoding
cat_features = df_adv.select_dtypes(exclude=['number']).columns
df_adv = pd.get_dummies(df_adv, columns=cat_features, drop_first=True, dtype=int)

print(f"Pozostałe braki w df_adv: {df_adv.isnull().sum().sum()}")

Pozostałe braki w df_adv: 0


In [11]:
# display(df_adv)

In [12]:
def evaluate_model(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2137)
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    return mae, r2

In [13]:
X_base = df_base.drop('survived', axis=1)
y_base = df_base['survived']

X_adv = df_adv.drop('survived', axis=1)
y_adv = df_adv['survived']

mae_base, r2_base = evaluate_model(X_base, y_base)
mae_adv, r2_adv = evaluate_model(X_adv, y_adv)

In [14]:
print("\n=== WYNIKI REGRESJI LINOWEJ ===")
print(f"Baseline model: MAE = {mae_base:.4f}, R2 = {r2_base:.4f}")
print(f"Advanced model: MAE = {mae_adv:.4f}, R2 = {r2_adv:.4f}")


=== WYNIKI REGRESJI LINOWEJ ===
Baseline model: MAE = 0.3085, R2 = 0.3000
Advanced model: MAE = 0.3031, R2 = 0.3177


In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

def evaluate_model_log(X, y):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2137)
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return acc

In [16]:
acc_base = evaluate_model_log(X_base, y_base)
acc_adv = evaluate_model_log(X_adv, y_adv)

print("\n=== WYNIKI REGRESJI LOGISTYCZNEJ ===")
print(f"Baseline model: Accuracy = {acc_base:.4f}")
print(f"Advanced model: Accuracy = {acc_adv:.4f}")


=== WYNIKI REGRESJI LOGISTYCZNEJ ===
Baseline model: Accuracy = 0.7672
Advanced model: Accuracy = 0.7977


Na internecje znalazłem, że wyniki >80 są już uznawane za bardzo dobre dla tego dataseta, więc skoro taka prosta regresja jest tak blisko to chyba nieźle

TUTAJ ZACZYNA SIĘ ZADANIE 1 Z LISTY 5, POWYŻEJ BYŁO ZADANIE 5 Z LISTY 4

In [17]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.metrics import accuracy_score
from sklearn.compose import ColumnTransformer

In [18]:
import warnings
warnings.filterwarnings('ignore') # bo jakieś błedy wyskakują

In [19]:
df = df_adv
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 14 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   pclass      1309 non-null   int64  
 1   survived    1309 non-null   int64  
 2   sex         1309 non-null   int64  
 3   age         1309 non-null   float64
 4   sibsp       1309 non-null   int64  
 5   parch       1309 non-null   int64  
 6   fare        1309 non-null   float64
 7   has_cabin   1309 non-null   int64  
 8   embarked_Q  1309 non-null   int64  
 9   embarked_S  1309 non-null   int64  
 10  title_Miss  1309 non-null   int64  
 11  title_Mr    1309 non-null   int64  
 12  title_Mrs   1309 non-null   int64  
 13  title_Rare  1309 non-null   int64  
dtypes: float64(2), int64(12)
memory usage: 143.3 KB


In [20]:
df.head()

,pclass,survived,sex,age,sibsp,parch,fare,has_cabin,embarked_Q,embarked_S,title_Miss,title_Mr,title_Mrs,title_Rare
0,1,1,0,29.00,0,0,211.3375,1,0,1,1,0,0,0
1,1,1,1,0.92,1,2,151.5500,1,0,1,0,0,0,0
2,1,0,0,2.00,1,2,151.5500,1,0,1,1,0,0,0
3,1,0,1,30.00,1,2,151.5500,1,0,1,0,1,0,0
4,1,0,0,25.00,1,2,151.5500,1,0,1,0,0,1,0


In [23]:
df["parch"].value_counts()

parch
0    1002
1     170
2     113
3       8
4       6
5       6
6       2
9       2
Name: count, dtype: int64

In [ ]:
target_col = 'survived'
# zmienne ciągłe, które podzielimy na grupy 
continuous_vars = ['age', 'fare'] # age też mógłby być od razu kategoryczny bo to są inty, ale możeby spówbować to trochę inaczej pogrupować
# pozostałe zmienne są już dyskretne ~ 0/1 oprócz parch, które jest 0-9, ale to i tak niewiele
rest_vars = [col for col in df.columns if col not in continuous_vars + [target_col]]

X = df[continuous_vars + rest_vars]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=2137, stratify=y # stratify - te same propocje klas itp
)

In [ ]:
k_values = [2, 5, 10, 25]
strategies = {
    'uniform': 'Równe części',
    'quantile': 'Równoliczne części',
    'kmeans': 'Klasteryzacja (K-Means)'
}

results = []

for strategy_name, strategy_desc in strategies.items():
    for k in k_values:
        
        binner = KBinsDiscretizer(n_bins=k, encode='ordinal', strategy=strategy_name)  # k to ilość kubełków, dzielimy wszystkie zmienne ciągłe na taką samą liczbę kubełków
        # ‘ordinal’: Return the bin identifier encoded as an integer value.
        # binner tnie na kawałki, według strategii
        
        preprocessor = ColumnTransformer(
            transformers=[
                ('binning', binner, continuous_vars), # podzieli ciągłe według strategi binnera
                ('pass', 'passthrough', rest_vars) # resztę olewa 
            ]
        )
        
        X_train_binned = preprocessor.fit_transform(X_train).astype(int) 
        X_test_binned = preprocessor.transform(X_test).astype(int)

        clf = CategoricalNB() # classifier
        clf.fit(X_train_binned, y_train)
            
        y_pred = clf.predict(X_test_binned)
        acc = accuracy_score(y_test, y_pred)
        
        results.append({
            'Metoda': strategy_desc,
            'Liczba grup (k)': k,
            'Dokładność (Accuracy)': acc
        })
            
results_df = pd.DataFrame(results)
pivot_table = results_df.pivot(index='Metoda', columns='Liczba grup (k)', values='Dokładność (Accuracy)')

pivot_table.round(4)

Liczba grup (k),2,5,10,25
Metoda,,,,
Klasteryzacja (K-Means),0.7786,0.7735,0.7659,0.7761
Równe części,0.7761,0.7659,0.7659,0.7710
Równoliczne części,0.7735,0.7710,0.7634,0.7837
